# RuBioRoBERTa в Google Colab

Трансформер на GPU: этап 6 (без файнтюна) и этап 7 (файнтюн + Optuna).

**Перед запуском:**
1. Загрузить папку проекта `anamnes` на Google Drive (с папками `src/` и `data/`).
2. Runtime → Change runtime type → GPU.
3. В ячейке конфигурации указать путь к проекту на Drive.

In [ ]:
!nvidia-smi

In [ ]:
!pip install -q transformers accelerate peft optuna

In [ ]:
import sys
from pathlib import Path

from google.colab import drive
drive.mount("/content/drive")

PROJECT_DIR = "/content/drive/MyDrive/anamnes"
assert Path(PROJECT_DIR, "src").exists(), "проверь PROJECT_DIR: нет папки src"
assert Path(PROJECT_DIR, "data").exists(), "проверь PROJECT_DIR: нет папки data"
if PROJECT_DIR not in sys.path:
    sys.path.insert(0, PROJECT_DIR)

## Этап 6. Без файнтюна (frozen + LogReg)

Эмбеддинги кэшируются в `models/` на Drive — повторный запуск мгновенный.

In [ ]:
from src.transformer_frozen import main as frozen_main
frozen_main("dev")

## Этап 7. Файнтюн (LoRA + взвешенный loss)

Один прогон с дефолтными гиперпараметрами для проверки, что всё учится.

In [ ]:
from src.transformer_ft import train_once
train_once(eval_split="dev", use_lora=True, save=True)

## Optuna (HPO)

Запускать после успешного одиночного прогона. Долго — начни с небольшого числа трайлов.

In [ ]:
from src.transformer_ft import run_optuna
study = run_optuna(n_trials=20, eval_split="dev", use_lora=True)

## Итоговые метрики

In [ ]:
import json
print(json.dumps(json.load(open(Path(PROJECT_DIR, "reports", "metrics.json"))), ensure_ascii=False, indent=2))